# UK interest rates across time and maturity with GeDS

This notebook introduces the Python interface to [GeDS](https://github.com/emilioluissaenzguillen/GeDS) through two related questions: how did the UK 10-year nominal spot rate change over time, and how did the *whole term structure* change? GeDS fits the curves and surface through its R implementation; Python handles data preparation and visualisation.

The source is the Bank of England's [monthly government nominal yield-curve archive](https://www.bankofengland.co.uk/statistics/yield-curves). These are **already estimated zero-coupon spot rates**, not raw gilt prices. We are describing published rates, not replacing the Bank's curve-construction method or forecasting future rates. The source archive can be revised; results may change if the Bank updates it. Rates below are in percentage points.

## Setup

Install R and the compatible GeDS 0.3.6 GitHub build as described in the [package README](https://github.com/emilioluissaenzguillen/GeDS-python#requirements). In your Python environment, run `python -m pip install "geds-python[plot]" openpyxl notebook`. Run `python -m geds.check` if GeDS cannot find R. The data download requires internet access.

In [ ]:
from io import BytesIO
from urllib.request import Request, urlopen
from zipfile import ZipFile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from geds import GeDSRegressor

ARCHIVE_URL = (
    'https://www.bankofengland.co.uk/-/media/boe/files/statistics/'
    'yield-curves/glcnominalmonthedata.zip'
)
WORKBOOK = 'GLC Nominal month end data_2016 to 2024.xlsx'
START, END = '2020-01-01', '2024-12-31'
MATURITIES = [2.0, 4.0, 6.0, 8.0, 10.0, 12.0, 15.0, 20.0]

request = Request(ARCHIVE_URL, headers={'User-Agent': 'Mozilla/5.0'})
with urlopen(request, timeout=60) as response:
    archive_bytes = response.read()
with ZipFile(BytesIO(archive_bytes)) as archive:
    with archive.open(WORKBOOK) as workbook:
        sheet = pd.read_excel(workbook, sheet_name='4. spot curve', header=None)

maturity_headers = pd.to_numeric(sheet.iloc[3, 1:], errors='coerce')
columns = [int(np.argmin(abs(maturity_headers.to_numpy() - m))) + 1 for m in MATURITIES]
assert all(abs(float(maturity_headers.iloc[c - 1]) - m) < 1e-6
           for c, m in zip(columns, MATURITIES))
wide = sheet.iloc[5:, [0, *columns]].copy()
wide.columns = ['date', *MATURITIES]
wide['date'] = pd.to_datetime(wide['date'], errors='coerce')
wide = wide.loc[wide['date'].between(START, END)].set_index('date').sort_index()
wide = wide.apply(pd.to_numeric, errors='coerce')
wide = wide.dropna(subset=MATURITIES)
assert len(wide) == 60 and wide.index.is_unique, 'Check the source layout or missing months.'
print(f'{len(wide)} month-end curves from {wide.index.min():%Y-%m} to {wide.index.max():%Y-%m}')
wide.head()

## 1. One maturity: the 10-year rate over time

We first hold maturity fixed at ten years. GeDS selects spline knots from the observed path, giving the fitted line room to bend where the historical series changes shape. This is descriptive smoothing, not a prediction of next month's yield.

In [ ]:
origin = wide.index.min()
time_years = (wide.index - origin).days.to_numpy() / 365.25
X_10y = pd.DataFrame({'time_years': time_years})
model_10y = GeDSRegressor(order=3).fit(X_10y, wide[10.0].to_numpy())
fitted_10y = model_10y.predict(X_10y)
knots_10y = np.asarray(model_10y.knots_, dtype=float).ravel()

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.scatter(wide.index, wide[10.0], s=24, alpha=0.75, label='Bank of England')
ax.plot(wide.index, fitted_10y, color='#cc4c02', lw=2.3, label='GeDS fit')
for knot in knots_10y:
    ax.axvline(origin + pd.to_timedelta(knot * 365.25, unit='D'),
               color='#cc4c02', alpha=0.22, lw=1)
ax.set(title='UK 10-year nominal spot rate', ylabel='Rate (% per year)', xlabel='Date')
ax.legend()
fig.tight_layout()
print(f'Internal knots: {len(knots_10y)}')
plt.show()

## 2. Two inputs: time and maturity

Each row now records an observation date, a maturity, and the corresponding published spot rate. A single joint bivariate GeDS term describes the changing shape of the curve. To make a modest interpolation check, we withhold complete months at regular intervals while retaining the first and last month in training. **This is not a forward-in-time forecasting test.**

In [ ]:
long = wide.rename_axis('date').reset_index().melt(
    id_vars='date', var_name='maturity_years', value_name='yield_pct'
)
long['maturity_years'] = long['maturity_years'].astype(float)
long['time_years'] = (long['date'] - origin).dt.days / 365.25
features = ['time_years', 'maturity_years']
held_out_dates = wide.index[5:-1:5]
is_holdout = long['date'].isin(held_out_dates)
train, test = long.loc[~is_holdout], long.loc[is_holdout]
surface = GeDSRegressor(spline_features=features, order=3).fit(
    train[features], train['yield_pct']
)
test_predictions = surface.predict(test[features])
geds_mae = np.mean(np.abs(test['yield_pct'].to_numpy() - test_predictions))

# Straight-line interpolation between adjacent *training* months at each maturity.
baseline = np.empty(len(test))
for maturity in MATURITIES:
    train_m = train.loc[train['maturity_years'] == maturity].sort_values('time_years')
    positions = np.flatnonzero(test['maturity_years'].to_numpy() == maturity)
    baseline[positions] = np.interp(
        test.iloc[positions]['time_years'],
        train_m['time_years'], train_m['yield_pct']
    )
baseline_mae = np.mean(np.abs(test['yield_pct'].to_numpy() - baseline))
print(f'Held-out months: {len(held_out_dates)}; observations: {len(test)}')
print(f'Mean absolute error, GeDS: {geds_mae:.3f} percentage points')
print(f'Mean absolute error, linear interpolation: {baseline_mae:.3f} percentage points')

In [ ]:
all_predictions = surface.predict(long[features])
predicted = long.assign(fitted=all_predictions).pivot(
    index='date', columns='maturity_years', values='fitted'
).loc[wide.index, MATURITIES]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), layout='constrained')
image = axes[0].imshow(
    predicted.to_numpy().T, aspect='auto', origin='lower', cmap='viridis',
    extent=[0, len(wide) - 1, 0, len(MATURITIES) - 1]
)
axes[0].set(
    title='GeDS fitted UK spot-rate surface', xlabel='Observation date',
    ylabel='Maturity (years)',
)
axes[0].set_xticks(np.linspace(0, len(wide) - 1, 5),
                  [wide.index[int(i)].strftime('%Y') for i in np.linspace(0, len(wide) - 1, 5)])
axes[0].set_yticks(range(len(MATURITIES)), [str(int(m)) for m in MATURITIES])
fig.colorbar(image, ax=axes[0], label='Rate (% per year)')

for date in [wide.index[12], wide.index[32], wide.index[48]]:
    axes[1].plot(MATURITIES, predicted.loc[date], marker='o', label=f'{date:%b %Y}')
axes[1].set(title='Fitted yield-curve slices', xlabel='Maturity (years)',
            ylabel='Rate (% per year)')
axes[1].legend()
plt.show()

## Interpretation and limits

The heatmap shows how both the level and the maturity profile of published UK nominal spot rates evolved. The held-out comparison asks how well the fitted surface reconstructs *interior months*; linear interpolation is a deliberately simple but relevant benchmark, and GeDS need not beat it. The knots and fit should be interpreted as features of this chosen sample and model settings, not as economically identified regime breaks.

This is not a pricing or trading model. GeDS does not impose absence of arbitrage on the fitted term structure, and the Bank's inputs are themselves fitted curves. Avoid extrapolating the surface beyond observed dates or maturities. See the Bank's [methodology notes](https://www.bankofengland.co.uk/statistics/yield-curves/terminology-and-concepts) for how its curves are constructed.